# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import itertools
from sklearn.metrics import accuracy_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv("day-of-week-not-scaled.csv")
df2 = pd.read_csv("../ex00/dayofweek.csv")
df['dayofweek'] = df2['dayofweek']
print(df.shape)
print(df.dtypes.to_list())
df.head()

(1686, 44)
[dtype('int64'), dtype('int64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64')]


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
random_state=21
test_size=0.2
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=random_state, test_size=test_size, stratify=y)
print('X_train shape: ', X_train.shape)
print('y_train shape: ', y_train.shape)
print('X_test shape : ', X_test.shape)
print('y_test shape : ', y_test.shape)

X_train shape:  (1348, 43)
y_train shape:  (1348,)
X_test shape :  (338, 43)
y_test shape :  (338,)


## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
model1 = SVC(random_state=random_state, probability=True)

params = {'kernel' : ['linear', 'rbf', 'sigmoid'],
          'C' : [0.01, 0.1, 1, 1.5, 5, 10],
          'gamma' : ['scale', 'auto'],
          'class_weight' : ['balanced', None]}

SVM_grid = GridSearchCV(estimator=model1, param_grid=params, n_jobs=-1)
SVM_grid.fit(X_train, y_train)

GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [5]:
SVM_grid.best_params_

{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}

In [6]:
results = pd.DataFrame(SVM_grid.cv_results_)

columns_to_show = ['param_kernel',
                   'param_C',
                   'param_gamma',
                   'param_class_weight', 
                   'mean_test_score']

results_table = results[columns_to_show].sort_values(by='mean_test_score', ascending=True)

results_table.rename(columns={'mean_test_score' : 'rank_test_score'}, inplace=True)
results_table

,param_kernel,param_C,param_gamma,param_class_weight,rank_test_score
29,sigmoid,1,auto,balanced,0.060088
17,sigmoid,0.1,auto,balanced,0.062310
41,sigmoid,1.5,auto,balanced,0.079380
65,sigmoid,10,auto,balanced,0.115693
53,sigmoid,5,auto,balanced,0.129792
...,...,...,...,...,...
60,linear,10,scale,balanced,0.721052
52,rbf,5,auto,balanced,0.808608
58,rbf,5,auto,None,0.816018
64,rbf,10,auto,balanced,0.863500


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [7]:
model2 = DecisionTreeClassifier(random_state=random_state)

params = {'max_depth' : range(1, 49),
          'criterion' : ['entropy', 'gini'],
          'class_weight' : ['balanced', None]}

tree_grid = GridSearchCV(estimator=model2, param_grid=params, n_jobs=-1)
tree_grid.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 49)})

In [8]:
tree_grid.best_params_

{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}

In [9]:
results = pd.DataFrame(tree_grid.cv_results_)

columns_to_show = ['param_max_depth',
                   'param_criterion',
                   'param_class_weight', 
                   'mean_test_score']

results_table = results[columns_to_show].sort_values(by='mean_test_score', ascending=True)

results_table.rename(columns={'mean_test_score' : 'rank_test_score'}, inplace=True)
results_table

,param_max_depth,param_criterion,param_class_weight,rank_test_score
0,1,entropy,balanced,0.286358
48,1,gini,balanced,0.286358
96,1,entropy,None,0.355330
144,1,gini,None,0.355330
50,3,gini,balanced,0.373906
...,...,...,...,...
84,37,gini,balanced,0.872372
95,48,gini,balanced,0.872372
69,22,gini,balanced,0.872378
72,25,gini,balanced,0.873854


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [10]:
model3 = RandomForestClassifier(random_state=random_state)

params = {'n_estimators' : [5, 10, 50, 100],
          'max_depth' : range(1, 49),
          'criterion' : ['entropy', 'gini'],
          'class_weight' : ['balanced', None]}

rf_grid = GridSearchCV(estimator=model3, param_grid=params)
rf_grid.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21),
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 49),
                         'n_estimators': [5, 10, 50, 100]})

In [11]:
rf_grid.best_params_

{'class_weight': 'balanced',
 'criterion': 'entropy',
 'max_depth': 24,
 'n_estimators': 100}

In [12]:
results = pd.DataFrame(rf_grid.cv_results_)

columns_to_show = ['param_n_estimators',
                   'param_max_depth',
                   'param_criterion',
                   'param_class_weight',
                   'mean_test_score']

results_table = results[columns_to_show].sort_values(by='mean_test_score', ascending=True)

results_table.rename(columns={'mean_test_score' : 'rank_test_score'}, inplace=True)
results_table

,param_n_estimators,param_max_depth,param_criterion,param_class_weight,rank_test_score
0,5,1,entropy,balanced,0.270794
192,5,1,gini,balanced,0.283390
196,5,2,gini,balanced,0.346419
4,5,2,entropy,balanced,0.353110
384,5,1,entropy,None,0.353832
...,...,...,...,...,...
699,100,31,gini,None,0.903547
310,50,30,gini,balanced,0.903549
115,100,29,entropy,balanced,0.904290
686,50,28,gini,None,0.904290


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [13]:
params = {'n_estimators' : [5, 10, 50, 100],
          'max_depth' : range(1, 49),
          'criterion' : ['entropy', 'gini'],
          'class_weight' : ['balanced', None]}

params_comb = list(itertools.product(params['n_estimators'], 
                                     params['max_depth'], 
                                     params['criterion'], 
                                     params['class_weight']))

results = []

for n_estimators, max_depth, criterion, class_weight in tqdm(params_comb):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        criterion=criterion,
        class_weight=class_weight,
        random_state=random_state,
        n_jobs=-1
    )

    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    
    results.append({
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'criterion': criterion,
        'class_weight': class_weight,
        'mean_accuracy': np.mean(scores),
        'std_accuracy': np.std(scores)
    })

100%|██████████| 768/768 [03:43<00:00,  3.44it/s]


In [14]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='mean_accuracy', ascending=False)

results_df

,n_estimators,max_depth,criterion,class_weight,mean_accuracy,std_accuracy
668,100,24,entropy,balanced,0.904293,0.012361
495,50,28,gini,None,0.904290,0.010961
688,100,29,entropy,balanced,0.904290,0.012156
502,50,30,gini,balanced,0.903549,0.012056
699,100,31,gini,None,0.903547,0.014380
...,...,...,...,...,...,...
1,5,1,entropy,None,0.353832,0.016467
4,5,2,entropy,balanced,0.353110,0.021165
6,5,2,gini,balanced,0.346419,0.029749
2,5,1,gini,balanced,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [15]:
best_model = RandomForestClassifier(random_state=random_state,
                                    n_estimators=100,
                                    max_depth=24,
                                    criterion='entropy',
                                    class_weight='balanced')

best_model.fit(X_train, y_train)
predict = best_model.predict(X_test)

acc = accuracy_score(predict, y_test)
print("Accuracy_score_4_best_model: ", acc)

Accuracy_score_4_best_model:  0.9260355029585798
